# ZeRO: Training State Has Owners

Prerequisites: gradients, Adam, data-parallel batch averaging, and byte counts.
Allow 60-90 minutes. You will build a memory model and simulate a sharded Adam
update, then prove it agrees with an unsharded baseline over multiple steps.

## The problem

A data-parallel training job has one logical model and multiple workers. Each
worker computes gradients from its own batch. Replicating every optimizer array
on every worker consumes memory without adding capacity to the learned model.
ZeRO removes replicas by assigning portions of training state to different
workers. Communication makes the required state available at the right time.

For this lab's mixed-precision recipe, each parameter needs:

| State | Bytes | Purpose |
|---|---:|---|
| Model weight | 2 | Forward/backward arithmetic |
| Gradient | 2 | Accumulated update signal |
| Master weight | 4 | Higher-precision optimizer update |
| Adam first moment | 4 | Smoothed gradient |
| Adam second moment | 4 | Smoothed squared gradient |

That is 16N bytes before activations or buffers. These are declared assumptions,
not a requirement that every framework uses 2-byte gradients.

## Derive each stage

Let R be workers and S be the largest shard, `ceil(N/R)` parameters.

| Stage | Weights | Gradients | Optimizer including master weights |
|---|---:|---:|---:|
| 0: replicated | 2N | 2N | 12N |
| 1 | 2N | 2N | 12S |
| 2 | 2N | 2S | 12S |
| 3 | 2S | 2S | 12S |

For one billion parameters and four workers, totals are 16, 7, 5.5, and 4 GB per
worker. The cluster still represents the same billion parameters. If N=5 and
R=2, shards contain 3 and 2 parameters; integer division alone undercounts the
larger worker. Our partitions are unpadded.

Activations, temporary gathered weights, communication buffers, fragmentation,
and checkpoints are separate. Stage 3's persistent total is not a peak-memory
prediction. More communication or recomputation can trade speed for capacity.

## Trace one optimizer step

Suppose rank gradients are `[2,4,6,8]` and `[4,6,8,10]`. Their equally sized
batches imply mean gradient `[3,5,7,9]`. A reduce-scatter combines gradients and
delivers `[3,5]` to owner 0 and `[7,9]` to owner 1. If batches have unequal token
counts, weight contributions by those counts; the exercise assumes equal batches.

Each owner keeps its own Adam history. For step t starting at 1:

```text
m = 0.9*m + 0.1*g
v = 0.999*v + 0.001*g*g
m_hat = m / (1 - 0.9**t)
v_hat = v / (1 - 0.999**t)
p_new = p - learning_rate * m_hat / (sqrt(v_hat) + 1e-8)
```

Initialize moments to zero only at the start of training. Resetting them each
step silently changes the optimizer. An all-gather can make updated weights
available for computation. Do not confuse gradient reduction with concatenating
different parameter slices.

## Your implementation

`zero_memory` returns component byte counts for the largest worker.
`sharded_adam_step` accepts disjoint parameter/moment shards and full per-rank
gradient vectors. Average, slice by actual shard lengths, update, and return new
lists without mutating inputs. Empty shards are valid when R>N.

The reference solution centrally averages arrays to model reduce-scatter. It
validates owner-update math, but it does not reduce this Python process's peak
memory or measure distributed communication. Do not label its runtime as a ZeRO
benchmark.

## Verification and experiments

The grader compares four successive updates with a separate dense Adam
calculation for 1, 2, 4, and 9 owners, including uneven and empty shards. It also
checks all memory stages and invalid inputs. In the notebook, change the number
of owners while keeping the same global gradient. Reconstructed parameters must
remain identical within numerical tolerance.

Deliberately replace the average with a sum or reset moments each step. Identify
the failing invariant. For a real distributed extension, instrument peak memory,
bytes communicated, and tokens/second using a documented DeepSpeed or FSDP
configuration; FSDP naming and policies are not interchangeable with all ZeRO
stages. That hardware extension is beyond this CPU simulator.

**Exit check:** Explain the difference between persistent state and peak memory;
reconstruct one parameter update; defend which stage fits a memory constraint.

Primary source: [Rajbhandari et al., ZeRO](https://arxiv.org/abs/1910.02054).


## Work first, then compare

CPU runtime is sufficient for the exercises. NumPy and PyTorch are available in Colab. Run the starter and grading cells after implementing the TODOs. An untouched starter is expected to fail. The reference solution below replaces the functions only when you deliberately run that cell. Then rerun the checks. No account, key, model download, or paid GPU is required.


In [ ]:
"""ZeRO: model-state accounting and a CPU simulation of an owner update.
This is a simulation of logical ranks, not a distributed runtime.
"""
import numpy as np

def zero_memory(params: int, ranks: int, stage: int) -> dict:
    """Largest rank's persistent state bytes: weights, gradients, optimizer.
    Assume 2-byte weights + 2-byte gradients + 12-byte master/Adam state.
    Stage 0 replicates all; 1 shards optimizer; 2 also gradients; 3 also weights.
    Uneven partitions: largest shard has ceil(params/ranks) elements, no padding.
    params>=0, ranks>0, stage in 0..3; otherwise raise ValueError.
    Excludes activations, communication buffers, allocator and gather peaks.
    """
    raise NotImplementedError

def sharded_adam_step(params, rank_gradients, moments, variances, step, lr=0.01):
    """One logical sharded optimizer step, returning (params, moments, variances).
    Each argument except step/lr is a list of NumPy arrays, one per rank.
    params/moments/variances: disjoint 1D shards, possibly unequal or empty.
    rank_gradients: one FULL gradient vector per rank, equally sized batches.
    Average gradients across ranks, scatter slices to owners, then update each
    owner's Adam moments with beta1=.9, beta2=.999 and bias correction at step>=1.
    eps=1e-8. Do not mutate inputs. Return three lists of updated shards.
    """
    raise NotImplementedError


In [ ]:
import numpy as np
assert [sum(zero_memory(1000, 4, s).values()) for s in range(4)] == [16000, 7000, 5500, 4000]
assert zero_memory(5, 2, 3) == {'weights': 6, 'gradients': 6, 'optimizer': 36}
assert sum(zero_memory(100, 1, 3).values()) == 1600
assert sum(zero_memory(0, 3, 2).values()) == 0
for _args in [(-1, 2, 0), (1, 0, 0), (1, 2, 4)]:
    try:
        zero_memory(*_args)
    except ValueError:
        pass
    else:
        raise AssertionError('invalid memory inputs must fail')
_rng = np.random.default_rng(8)
for _ranks in [1, 2, 4, 9]:
    _full = _rng.normal(size=7)
    _shards = [a.copy() for a in np.array_split(_full, _ranks)]
    _ms = [np.zeros_like(a) for a in _shards]
    _vs = [np.zeros_like(a) for a in _shards]
    _m, _v = np.zeros(7), np.zeros(7)
    for _step in range(1, 5):
        _grads = [_rng.normal(size=7) for _ in range(_ranks)]
        _before = [a.copy() for a in _shards]
        _new, _ms, _vs = sharded_adam_step(_shards, _grads, _ms, _vs, _step, .03)
        for _a, _b in zip(_before, _shards):
            np.testing.assert_array_equal(_a, _b)
        _shards = _new
        _g = np.mean(_grads, axis=0)
        _m = .9 * _m + .1 * _g
        _v = .999 * _v + .001 * _g ** 2
        _full -= .03 * (_m / (1 - .9 ** _step)) / (np.sqrt(_v / (1 - .999 ** _step)) + 1e-8)
        np.testing.assert_allclose(np.concatenate(_shards), _full, atol=1e-12)
        np.testing.assert_allclose(np.concatenate(_ms), _m, atol=1e-12)
        np.testing.assert_allclose(np.concatenate(_vs), _v, atol=1e-12)
print('ZeRO accounting and four sharded Adam steps agree with the unsharded baseline.')


## Worked reference solution

Compare the state transitions and invariants with your implementation. Passing output tests alone does not establish memory efficiency or reproduce the paper's performance results.


In [ ]:
import numpy as np

def zero_memory(params, ranks, stage):
    if params < 0 or ranks <= 0 or stage not in (0, 1, 2, 3):
        raise ValueError('invalid parameter count, rank count, or stage')
    shard = (params + ranks - 1) // ranks
    return {'weights': 2 * (shard if stage >= 3 else params),
            'gradients': 2 * (shard if stage >= 2 else params),
            'optimizer': 12 * (shard if stage >= 1 else params)}

def sharded_adam_step(params, rank_gradients, moments, variances, step, lr=0.01):
    # This centralized average models reduce-scatter, not its communication cost.
    gradient = np.mean(rank_gradients, axis=0)
    new_params, new_m, new_v = [], [], []
    start = 0
    for p, m, v in zip(params, moments, variances):
        g = gradient[start:start + len(p)]
        m = .9 * m + .1 * g
        v = .999 * v + .001 * g * g
        update = (m / (1 - .9 ** step)) / (np.sqrt(v / (1 - .999 ** step)) + 1e-8)
        new_params.append(p - lr * update)
        new_m.append(m)
        new_v.append(v)
        start += len(p)
    return new_params, new_m, new_v


In [ ]:
import numpy as np
assert [sum(zero_memory(1000, 4, s).values()) for s in range(4)] == [16000, 7000, 5500, 4000]
assert zero_memory(5, 2, 3) == {'weights': 6, 'gradients': 6, 'optimizer': 36}
assert sum(zero_memory(100, 1, 3).values()) == 1600
assert sum(zero_memory(0, 3, 2).values()) == 0
for _args in [(-1, 2, 0), (1, 0, 0), (1, 2, 4)]:
    try:
        zero_memory(*_args)
    except ValueError:
        pass
    else:
        raise AssertionError('invalid memory inputs must fail')
_rng = np.random.default_rng(8)
for _ranks in [1, 2, 4, 9]:
    _full = _rng.normal(size=7)
    _shards = [a.copy() for a in np.array_split(_full, _ranks)]
    _ms = [np.zeros_like(a) for a in _shards]
    _vs = [np.zeros_like(a) for a in _shards]
    _m, _v = np.zeros(7), np.zeros(7)
    for _step in range(1, 5):
        _grads = [_rng.normal(size=7) for _ in range(_ranks)]
        _before = [a.copy() for a in _shards]
        _new, _ms, _vs = sharded_adam_step(_shards, _grads, _ms, _vs, _step, .03)
        for _a, _b in zip(_before, _shards):
            np.testing.assert_array_equal(_a, _b)
        _shards = _new
        _g = np.mean(_grads, axis=0)
        _m = .9 * _m + .1 * _g
        _v = .999 * _v + .001 * _g ** 2
        _full -= .03 * (_m / (1 - .9 ** _step)) / (np.sqrt(_v / (1 - .999 ** _step)) + 1e-8)
        np.testing.assert_allclose(np.concatenate(_shards), _full, atol=1e-12)
        np.testing.assert_allclose(np.concatenate(_ms), _m, atol=1e-12)
        np.testing.assert_allclose(np.concatenate(_vs), _v, atol=1e-12)
print('ZeRO accounting and four sharded Adam steps agree with the unsharded baseline.')


## Experiment and interpretation

Predict the result before running. Change one variable, repeat, and record what changed. Separate measured outcomes from assumptions.


In [ ]:
for ranks in [1, 2, 4, 8]:
    for stage in range(4):
        state = zero_memory(1_000_000_000, ranks, stage)
        print(ranks, stage, {k: v/1e9 for k,v in state.items()})
print("These are persistent GB per logical rank, not measured process memory.")
